In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
print(os.path.exists("/content/drive/MyDrive/Colab Notebooks/IMDB Dataset.csv"))

True


In [6]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/IMDB Dataset.csv")
print(df.shape)
print(df.head())
print(df['sentiment'].value_counts())

(50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [14]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)       # remove HTML line breaks
    text = re.sub(r'[^a-z\s]', '', text)         # remove punctuation/numbers
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df['clean_review'] = df['review'].apply(clean_text)
print(df[['review', 'clean_review']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                        clean_review  
0  one reviewers mentioned watching oz episode yo...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically theres family little boy jake thinks...  
4  petter matteis love time money visually stunni...  


In [8]:
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

df['tokens'] = df['clean_review'].apply(word_tokenize)

all_words = [word for tokens in df['tokens'] for word in tokens]
from collections import Counter
word_freq = Counter(all_words)
print(word_freq.most_common(20))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


[('movie', 85236), ('film', 76035), ('one', 51480), ('like', 39066), ('good', 28913), ('even', 24586), ('would', 24036), ('time', 23972), ('really', 23014), ('see', 22641), ('story', 22521), ('well', 19235), ('much', 19110), ('get', 18245), ('bad', 18004), ('great', 17936), ('also', 17859), ('people', 17710), ('first', 17207), ('dont', 16976)]


In [9]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=5000)
X_bow = vectorizer.fit_transform(df['clean_review'])

print(X_bow.shape)
print(vectorizer.get_feature_names_out()[:20])

(50000, 5000)
['aaron' 'abandoned' 'abc' 'abilities' 'ability' 'able' 'absence' 'absent'
 'absolute' 'absolutely' 'absurd' 'abuse' 'abused' 'abusive' 'abysmal'
 'academy' 'accent' 'accents' 'accept' 'acceptable']


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_review'])

print(X_tfidf.shape)
print(X_tfidf[0].toarray())

(50000, 5000)
[[0. 0. 0. ... 0. 0. 0.]]


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, df['sentiment'], test_size=0.2, random_state=42)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_preds = nb_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_preds))
print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_preds))

Logistic Regression Accuracy: 0.8891
Naive Bayes Accuracy: 0.8514


In [17]:
X_test_indices = y_test.index
wrong_preds = X_test_indices[lr_preds != y_test]

for idx in wrong_preds[:5]:
    print("Review:", df.loc[idx, 'review'][:300])
    print("Actual:", df.loc[idx, 'sentiment'], "| Predicted:", lr_model.predict(X_tfidf[df.index.get_loc(idx)])[0])
    print("---")

Review: I really liked this Summerslam due to the look of the arena, the curtains and just the look overall was interesting to me for some reason. Anyways, this could have been one of the best Summerslam's ever if the WWF didn't have Lex Luger in the main event against Yokozuna, now for it's time it was ok 
Actual: positive | Predicted: negative
---
Review: Okay, I didn't get the Purgatory thing the first time I watched this episode. It seemed like something significant was going on that I couldn't put my finger on. This time those Costa Mesa fires on TV really caught my attention- and it helped that I was just writing an essay on Inferno! But let me s
Actual: positive | Predicted: negative
---
Review: The production quality, cast, premise, authentic New England (Waterbury, CT?) locale and lush John Williams score should have resulted in a 3-4 star collectors item. Unfortunately, all we got was a passable 2 star "decent" flick, mostly memorable for what it tried to do.........bring an 

## Project Summary: IMDB Sentiment Classifier

**Goal:** Classify movie reviews as positive/negative using text features.

**Pipeline:**
1. Cleaned text (lowercase, removed HTML tags, punctuation, stopwords)
2. Tokenized and verified cleaning via word frequency check
3. Converted text to numbers using Bag of Words and TF-IDF
4. Trained Logistic Regression and Naive Bayes on TF-IDF features

**Results:**
- Logistic Regression: 88.91% accuracy
- Naive Bayes: 85.14% accuracy
- LR outperformed NB, likely because NB assumes word independence while LR can weigh word combinations

**Limitations (from error analysis):**
- Model struggles with mixed-sentiment reviews (e.g., mostly positive but with some criticism)
- TF-IDF can't capture negation ("wasn't disappointed") or context — treats words independently
- This is a known gap vs. context-aware models like BERT